In [ ]:
# !mkdir src
# !pip install -q evaluate

In [ ]:
%%writefile src/config.py

hf_user_name = "amin-oj"
train_sample_size = 8 * 1024
validation_sample_size = 1 * 1024
dataset_dict = dict(path = "squad")
model_checkpoint = "bert-base-cased"
push_to_hub=False
eval_on_start = True
train_bs = 16
eval_bs = 16
num_train_epochs = 2
gacc_steps = 2
learning_rate = 2e-5
max_grad_norm = 1.0
lr_scheduler_type = "linear"
task = "QA"
max_seq_len = 384
qa_stride = 128
ckpt_name = f"{model_checkpoint.split("/")[-1]}-finetuned-{task}-{dataset_dict['path'].split("/")[-1]}-accelerate"

In [ ]:
%%writefile src/utils.py

from kaggle_secrets import UserSecretsClient
import os, torch

def hf_login():
    user_secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

def get_mp():
    bf16_supported = False
    if torch.cuda.is_available():
        try:
            bf16_supported = torch.cuda.is_bf16_supported()
        except Exception:
            bf16_supported = False

    mixed_precision = "bf16" if bf16_supported else "fp16"
    return mixed_precision

In [ ]:
%%writefile src/preprocess.py

import numpy as np
from src.config import max_seq_len, qa_stride

def find_answer_positions(sequence_ids, offset, start_char, end_char):
    if 1 not in sequence_ids:
        return np.int64(0), np.int64(0)
    context_start = sequence_ids.index(1)
    context_end = sequence_ids.index(None, context_start) -1
    offset = np.array(offset)
    
    # If the answer is not fully inside the context, label it (0, 0)
    if (offset[context_start][0] > end_char) or (offset[context_end][1] < start_char):
        return np.int64(0), np.int64(0)
    else:
        start_position = context_start + np.argmax(offset[context_start:context_end+1, 0] == start_char)
        end_position = context_start + np.argmax(offset[context_start:context_end+1, 1] == end_char)

    return start_position, end_position


def preprocess_training_examples(
    examples,
    tokenizer,
    max_length = max_seq_len,
    stride = qa_stride
):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)
    
        start_position, end_position = find_answer_positions(
            sequence_ids, offset, start_char, end_char
        )
    
        start_positions.append(start_position)
        end_positions.append(end_position)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    
    return inputs


def preprocess_validation_examples(
    examples,
    tokenizer,
    max_length = max_seq_len,
    stride = qa_stride
):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []

    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])

        sequence_ids = inputs.sequence_ids(i)
        offset = inputs["offset_mapping"][i]
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
        ]

    inputs["example_id"] = example_ids
    return inputs

In [ ]:
%%writefile src/data.py

from collections import defaultdict
from datasets import load_dataset, DatasetDict
from src.config import (
    train_sample_size, validation_sample_size,
    dataset_dict, train_bs, eval_bs
)
from src.preprocess import (
    preprocess_training_examples,
    preprocess_validation_examples
)
from torch.utils.data import DataLoader
from transformers import default_data_collator

def load_raw_ds():
    train_ds = load_dataset(**dataset_dict, split = f"train[:{train_sample_size}]")
    valid_ds = load_dataset(**dataset_dict, split = f"train[:{validation_sample_size}]")
    raw_datasets = DatasetDict({
        "train": train_ds,
        "validation": valid_ds
    })
    return raw_datasets


def prep_datasets(raw_datasets, tokenizer):
    train_dataset = raw_datasets["train"].map(
        preprocess_training_examples,
        batched=True,
        remove_columns=raw_datasets["train"].column_names,
        fn_kwargs = {"tokenizer": tokenizer},
    )
    
    validation_dataset = raw_datasets["validation"].map(
        preprocess_validation_examples,
        batched=True,
        remove_columns=raw_datasets["validation"].column_names,
        fn_kwargs = {"tokenizer": tokenizer},
    )
    return train_dataset, validation_dataset

def create_dataloaders(train_dataset, validation_dataset):
    train_dataloader = DataLoader(
        train_dataset,
        shuffle=True,
        collate_fn=default_data_collator,
        batch_size=train_bs,
        drop_last=True,
    )
    eval_dataloader = DataLoader(
        validation_dataset,
        collate_fn=default_data_collator,
        batch_size=eval_bs
    )
    return train_dataloader, eval_dataloader

In [ ]:
%%writefile src/postprocess.py

import numpy as np
from collections import defaultdict

def postprocess_predictions(
    start_logits,
    end_logits,
    tokenized_eval_examples,
    eval_examples,
    n_best = 20,
    max_answer_length = 30
):
    example_to_features = defaultdict(list)
    for idx, feature in enumerate(tokenized_eval_examples):
        example_to_features[feature["example_id"]].append(idx)

    predicted_answers = []
    for example in eval_examples:
        example_id = example["id"]
        context = example["context"]
        answers = []

        # Loop through all tokenized_eval_examples associated with that example
        for feature_index in example_to_features[example_id]:
            start_logit = start_logits[feature_index]
            end_logit = end_logits[feature_index]
            offsets = tokenized_eval_examples[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()
            end_indexes = np.argsort(end_logit)[-1 : -n_best - 1 : -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    # Skip answers that are not fully in the context
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue
                    # Skip answers with a length that is either < 0 or > max_answer_length
                    if (
                        end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    answer = {
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "logit_score": start_logit[start_index] + end_logit[end_index],
                    }
                    answers.append(answer)

        # Select the answer with the best score
        if len(answers) > 0:
            best_answer = max(answers, key=lambda x: x["logit_score"])
            predicted_answers.append(
                {"id": example_id, "prediction_text": best_answer["text"]}
            )
        else:
            predicted_answers.append({"id": example_id, "prediction_text": ""})

    return predicted_answers

In [ ]:
%%writefile src/evaluation.py

import evaluate
import torch, math
from src.postprocess import postprocess_predictions
from tqdm.auto import tqdm
import numpy as np

def compute_metrics(
    predicted_answers,
    eval_examples
):
    metric = evaluate.load("squad")
    theoretical_answers = [
        {"id": ex["id"], "answers": ex["answers"]}
        for ex in eval_examples
    ]
    return metric.compute(
        predictions=predicted_answers,
        references=theoretical_answers
    )



def run_eval(
    model, eval_dataloader,
    accelerator,
    validation_dataset,
    raw_validation_dataset
):
    accelerator.print("Starting Evaluation Loop ....")
    model.eval()
    print(f"[rank={accelerator.process_index}] eval_dataloader batches: {len(eval_dataloader)}")
    
    start_logits = []
    end_logits = []
    
    for i, batch in enumerate(eval_dataloader):
        bs = batch["input_ids"].shape[0]
        with torch.no_grad():
            with accelerator.autocast():
                outputs = model(**batch)
        
        # gather both loss and batch size across processes
        start_logit_t = accelerator.gather_for_metrics(outputs.start_logits).cpu().numpy()
        end_logit_t   = accelerator.gather_for_metrics(outputs.end_logits).cpu().numpy()
        start_logits.append(start_logit_t)
        end_logits.append(end_logit_t)
        if (i+1)%10 == 0: accelerator.print(f"{i+1} eval batches processed")

    accelerator.print(f"{i+1} eval batches processed")
    start_logits = np.concatenate(start_logits)
    end_logits = np.concatenate(end_logits)
    start_logits = start_logits[: len(validation_dataset)]
    end_logits = end_logits[: len(validation_dataset)]

    predicted_answers = postprocess_predictions(
        start_logits,
        end_logits,
        validation_dataset,
        raw_validation_dataset
    )
    return compute_metrics(predicted_answers, raw_validation_dataset)

In [ ]:
%%writefile main.py

import os, torch, math
import numpy as np
from tqdm.auto import tqdm
from huggingface_hub import HfApi, create_repo
from transformers import AutoTokenizer
from transformers import AutoModelForQuestionAnswering
from torch.optim import AdamW
from accelerate import Accelerator
from accelerate.utils import set_seed as accl_set_seed

from src.config import *
from src.utils import hf_login, get_mp
from src.data import load_raw_ds, prep_datasets, create_dataloaders
from src.evaluation import run_eval
from transformers import get_scheduler



if __name__ == "__main__":
    
    accelerator = Accelerator(
        gradient_accumulation_steps=gacc_steps,
        mixed_precision=get_mp(),
    )
    # accl_set_seed(42)
    
    accelerator.print(f"checkpoint name: {ckpt_name}")
    accelerator.print(f"# GPUs: {accelerator.num_processes}")
    effective_bs = train_bs * gacc_steps * accelerator.num_processes
    accelerator.print(f"effective_batch_size: {effective_bs}")
    # HF Login and repo initiation
    with accelerator.main_process_first():
        hf_login()
    if accelerator.is_main_process:
        hf_api = HfApi()
        accelerator.print(f"HF user: {hf_api.whoami()['name']}")
        if push_to_hub:
            HF_REPO_ID = f"{hf_user_name}/{ckpt_name}"
            create_repo(repo_id=HF_REPO_ID, exist_ok=True)
            accelerator.print(f"HF repo: {HF_REPO_ID}")
    
    with accelerator.main_process_first():
        raw_datasets = load_raw_ds()
        
        train_ans_count = len(raw_datasets["train"].filter(lambda x: len(x["answers"]["text"]) != 1))
        assert train_ans_count==0, "all train records must have exactly 1 answer text."
        
        tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
        assert tokenizer.is_fast, "tokenizer must be fast."
        
        train_dataset, validation_dataset = prep_datasets(raw_datasets, tokenizer)
        train_dataset.set_format("torch")
        validation_dataset.set_format("torch")
        train_dataloader, eval_dataloader = create_dataloaders(
            train_dataset,
            validation_dataset.remove_columns(["example_id", "offset_mapping"])
        )
        model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)
    
    
    
    accelerator.print("initial number of records:=========")
    for split in raw_datasets:
        accelerator.print(f"{split}: {len(raw_datasets[split])}")
    accelerator.print("=================")
    accelerator.print(f"tokenized train records: {len(train_dataset)}")
    accelerator.print("=================")
    accelerator.print(f"tokenized validation records: {len(validation_dataset)}")
    accelerator.print("=================")
    
    # Preparing training components
    optimizer = AdamW(model.parameters(), lr=learning_rate)
    model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
        model, optimizer, train_dataloader, eval_dataloader
    )
    num_update_steps_per_epoch = math.ceil(len(train_dataloader) / gacc_steps)
    num_training_steps = num_train_epochs * num_update_steps_per_epoch
    logging_steps = max(1, num_update_steps_per_epoch // 4)
    eval_steps = num_update_steps_per_epoch
    
    accelerator.wait_for_everyone()
    rank = accelerator.process_index
    is_main = accelerator.is_main_process
    pid = f"[rank={rank} main={is_main}]"
    print(f"{pid}|train_len={len(train_dataloader)}|eval_len={len(eval_dataloader)}")
    print(f"{pid}|eval_steps={eval_steps}|train_opt_steps={num_training_steps}")
    accelerator.wait_for_everyone()
    
    lr_scheduler = get_scheduler(
        lr_scheduler_type,
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps,
    )
    # Training loop
    global_step = 0
    running_loss = 0.0
    running_mb = 0
    scheduler_steps = 0

    
    progress_bar = tqdm(
        total=num_training_steps,
        desc="Optimizer Updates",
        disable=not accelerator.is_main_process,
    )

    if eval_on_start:
        eval_logs = run_eval(
            model, eval_dataloader,
            accelerator,
            validation_dataset,
            raw_datasets["validation"]
        )
        accelerator.print(f"Step {global_step} eval:", eval_logs)

    for epoch in range(num_train_epochs):
        model.train()
        for step, batch in enumerate(train_dataloader):
            with accelerator.accumulate(model):
                with accelerator.autocast():
                    outputs = model(**batch)
                    loss = outputs.loss
                running_loss += loss.detach().float().item()
                running_mb += 1
                accelerator.backward(loss)
                do_sync = accelerator.sync_gradients
                if do_sync:
                    accelerator.clip_grad_norm_(model.parameters(), max_grad_norm)
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
                    lr_scheduler.step()
                    scheduler_steps += 1
            
            # We count "global_step" in terms of optimizer updates
            if do_sync:
                global_step += 1
                progress_bar.update(1)

                # TRAIN logging (main process only)
                if global_step % logging_steps == 0:
                    avg_train_loss = running_loss / max(1, running_mb)
                    lr_val = optimizer.param_groups[0]["lr"]
                    loss_t = torch.tensor([avg_train_loss], device=accelerator.device)
                    lr_t   = torch.tensor([lr_val], device=accelerator.device)
                    loss_all = accelerator.gather_for_metrics(loss_t).detach().cpu().tolist()
                    lr_all   = accelerator.gather_for_metrics(lr_t).detach().cpu().tolist()
                    for r, (l, lr_) in enumerate(zip(loss_all, lr_all)):
                        accelerator.print(
                            f"train_loss/rank_{r} = {l} | learning_rate/rank_{r} = {lr_}"
                        )
                    running_loss = 0.0
                    running_mb = 0
                # EVAL logging
                if (global_step % eval_steps == 0) or (global_step == num_training_steps):
                    eval_logs = run_eval(
                            model, eval_dataloader,
                            accelerator,
                            validation_dataset,
                            raw_datasets["validation"]
                        )
                    accelerator.print(f"Step {global_step} eval: {eval_logs}")

    if accelerator.is_main_process:
        accelerator.print("Scheduler steps:", scheduler_steps, "Expected:", num_training_steps)
    accelerator.wait_for_everyone()
    progress_bar.close()
    accelerator.end_training()
    accelerator.free_memory()

In [ ]:
! accelerate launch --num_processes 2 main.py